# Warm-up 3: Tokenization, Padding, and Masking

Этот notebook связан с guide [Tokens, Padding, Masking, and Decoder Shift](../guides/02_tokens_padding_and_decoder_shift.md).

Цель:
- руками собрать маленький словарь;
- построить `encoder_input`, `decoder_input`, `decoder_target`;
- увидеть mask и строгую метрику без тяжёлого обучения.


In [1]:
import numpy as np
import tensorflow as tf

source_sequences = [
    ['red', 'blue', 'green'],
    ['blue', 'yellow'],
    ['green', 'green', 'red'],
]
target_sequences = [list(reversed(seq)) for seq in source_sequences]

special_tokens = ['[PAD]', '[SOS]', '[EOS]']
vocab_tokens = sorted({token for seq in source_sequences for token in seq})
vocab = special_tokens + vocab_tokens
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

PAD_ID = token_to_id['[PAD]']
SOS_ID = token_to_id['[SOS]']
EOS_ID = token_to_id['[EOS]']

print('vocab:', vocab)
print('token_to_id:', token_to_id)


2026-03-24 10:58:28.269140: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-24 10:58:28.269453: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 10:58:28.271272: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-24 10:58:28.276188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774339108.285316  190158 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774339108.28

vocab: ['[PAD]', '[SOS]', '[EOS]', 'blue', 'green', 'red', 'yellow']
token_to_id: {'[PAD]': 0, '[SOS]': 1, '[EOS]': 2, 'blue': 3, 'green': 4, 'red': 5, 'yellow': 6}


In [2]:
encoder_ids = [[token_to_id[token] for token in seq] for seq in source_sequences]
target_ids = [[token_to_id[token] for token in seq] for seq in target_sequences]

decoder_input_ids = [[SOS_ID] + seq for seq in target_ids]
decoder_target_ids = [seq + [EOS_ID] for seq in target_ids]

encoder_input = tf.keras.utils.pad_sequences(encoder_ids, padding='post', value=PAD_ID)
decoder_input = tf.keras.utils.pad_sequences(decoder_input_ids, padding='post', value=PAD_ID)
decoder_target = tf.keras.utils.pad_sequences(decoder_target_ids, padding='post', value=PAD_ID)

print('encoder_input shape :', encoder_input.shape)
print(encoder_input)
print()
print('decoder_input shape :', decoder_input.shape)
print(decoder_input)
print()
print('decoder_target shape:', decoder_target.shape)
print(decoder_target)


encoder_input shape : (3, 3)
[[5 3 4]
 [3 6 0]
 [4 4 5]]

decoder_input shape : (3, 4)
[[1 4 3 5]
 [1 6 3 0]
 [1 5 4 4]]

decoder_target shape: (3, 4)
[[4 3 5 2]
 [6 3 2 0]
 [5 4 4 2]]


In [3]:
row = 0
print('source tokens :', source_sequences[row])
print('target tokens :', target_sequences[row])
print('decoder_input :', [id_to_token[idx] for idx in decoder_input[row] if idx != PAD_ID])
print('decoder_target:', [id_to_token[idx] for idx in decoder_target[row] if idx != PAD_ID])


source tokens : ['red', 'blue', 'green']
target tokens : ['green', 'blue', 'red']
decoder_input : ['[SOS]', 'green', 'blue', 'red']
decoder_target: ['green', 'blue', 'red', '[EOS]']


In [4]:
mask = (decoder_target != PAD_ID).astype(np.float32)

fake_preds = decoder_target.copy()
fake_preds[1, 1] = token_to_id['red']

token_accuracy = ((fake_preds == decoder_target) * mask).sum() / mask.sum()
exact_match = np.all((fake_preds == decoder_target) | (mask == 0), axis=1).mean()

print('mask shape:', mask.shape)
print(mask)
print()
print(f'token_accuracy = {token_accuracy:.3f}')
print(f'exact_match    = {exact_match:.3f}')
print('One wrong token is enough to break exact_match for that sequence.')


mask shape: (3, 4)
[[1. 1. 1. 1.]
 [1. 1. 1. 0.]
 [1. 1. 1. 1.]]

token_accuracy = 0.909
exact_match    = 0.667
One wrong token is enough to break exact_match for that sequence.


In [5]:
embedding = tf.keras.layers.Embedding(input_dim=len(vocab), output_dim=8, mask_zero=True)
embedded = embedding(encoder_input)
embedded_mask = embedding.compute_mask(encoder_input)

print('embedded shape:', embedded.shape)
print('embedding mask:')
print(embedded_mask.numpy().astype(np.int32))


embedded shape: (3, 3, 8)
embedding mask:
[[1 1 1]
 [1 1 0]
 [1 1 1]]


E0000 00:00:1774339109.999190  190158 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774339110.001800  190158 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Что взять с собой дальше

- `decoder_input` и `decoder_target` обязаны быть сдвинуты.
- `PAD` нужен для выравнивания batch, а mask — чтобы padded позиции не портили обучение и метрики.
- `Embedding(mask_zero=True)` удобно сигнализирует downstream-слоям, где лежит padding.
